# Session 10: Using Ragas to Evaluate a RAG Application built with LangChain and LangGraph

In the following notebook, we'll be looking at how [Ragas](https://github.com/explodinggradients/ragas) can be helpful in a number of ways when looking to evaluate your RAG applications!

While this example is rooted in LangChain/LangGraph - Ragas is framework agnostic (you don't even need to be using a framework!).

## 🤝 Breakout Room #1
  - Task 1: Installing Required Libraries
  - Task 2: Set Environment Variables
  - Task 3: Synthetic Dataset Generation for Evaluation using Ragas
  - Task 4: Construct our RAG application
  - Task 5: Evaluating our Application with Ragas
  - Task 6: Making Adjustments and Re-Evaluating
  - ***Activity #1: Implement a Different Reranking Strategy***


## Task 1: Installing Required Libraries

If you have not already done so, install the required libraries using the uv package manager:
``` bash

uv sync

```


## Task 2: Set Environment Variables:

We'll also need to provide our API keys.
> NOTE: In addition to OpenAI's models, this notebook will be using Cohere's Reranker - please be sure to [sign-up for an API key!](https://docs.cohere.com/reference/about)

You have two options for supplying your API keys in this session:
- Use environment variables (see Prerequisite #2 in the README.md)
- Provide them via a prompt when the notebook runs

The following code will load all of the environment variables in your `.env`. Then, it checks for the two API keys we need. If they are not there, it will prompt you to provide them.

First, OpenAI's for our LLM/embedding model combination!

Second, Cohere's for our reranking


In [44]:
import os
from getpass import getpass
from dotenv import load_dotenv



#load_dotenv()
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass("Please enter your Cohere API key!")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")


print("!!!!")
os.environ["OPENAI_API_KEY"]
print("!!!!")
os.environ["COHERE_API_KEY"]





!!!!
!!!!


'6F5C0AalZCOV8m9cktyuo0RMpw2XTDYtUu9ftsrA'

## Task 3: Synthetic Dataset Generation for Evaluation using Ragas

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using the Health & Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, and stress management.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [45]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [46]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

In [47]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/4 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/9 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [48]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How can I perform a Chest Opener to relieve ne...,[The Personal Wellness Guide A Comprehensive R...,"To perform a Chest Opener, clasp your hands be...",single_hop_specifc_query_synthesizer
1,How do I do Knee-to-Chest Stretch for help wit...,[The Personal Wellness Guide A Comprehensive R...,"To do the Knee-to-Chest Stretch, lie on your b...",single_hop_specifc_query_synthesizer
2,how do i do partial crunches for back pain?,[The Personal Wellness Guide A Comprehensive R...,"Lie on your back with knees bent, cross arms o...",single_hop_specifc_query_synthesizer
3,What Celsius temperature range is recommended ...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,The recommended temperature range for an optim...,single_hop_specifc_query_synthesizer
4,What role does magnesium play in improving sle...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,Magnesium supplements are mentioned as a natur...,single_hop_specifc_query_synthesizer
5,What is Cognitive Behavioral Therapy for Insom...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,Cognitive Behavioral Therapy for Insomnia (CBT...,single_hop_specifc_query_synthesizer
6,What actionable strategies does Chapter 14 rec...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 14 suggests several actionable strateg...,single_hop_specifc_query_synthesizer
7,"According to Chapter 14, what are the recommen...",[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 14 emphasizes that how you start your ...,single_hop_specifc_query_synthesizer
8,What actionable strategies for improving diges...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 17 recommends several strategies for s...,single_hop_specifc_query_synthesizer
9,Why shud I avoid caffine after 2 PM?,[hour before bed - No caffeine after 2 PM - Co...,No caffeine after 2 PM is recommended to suppo...,single_hop_specifc_query_synthesizer


## Task 4: Construct our RAG application

Now we'll construct our LangChain RAG, which we will be evaluating using the above created test data!

### R - Retrieval

Let's start with building our retrieval pipeline, which will involve loading the same data we used to create our synthetic test set above.

> NOTE: We need to use the same data - as our test set is specifically designed for this data.

In [49]:
loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()

Now that we have our data loaded, let's split it into chunks!

In [50]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

447

### ❓ Question #1:

What is the purpose of the `chunk_overlap` parameter in the `RecursiveCharacterTextSplitter`?

##### Answer:
The chunk_overlap parameter in RecursiveCharacterTextSplitter controls how many characters are shared between consecutive chunks when a document is split.

What it does
1. Without overlap: Each chunk ends exactly where the next begins. Text at the boundaries can be cut off (e.g., mid‑word or mid‑phrase).
2. With overlap: Each chunk extends slightly into the next one. For example, with overlap 50, chunk 1 might end with "...and the result" and chunk 2 might start with "the result was..." so both chunks share "and the result" or "the result".

Why it’s used
1. Context continuity – Related phrases or sentences are less likely to be split across boundaries.
2. Better retrieval – Overlap helps ensure that important phrases at chunk edges appear in at least one full chunk.
3. Fewer broken sentences – Less risk of cutting mid‑sentence when splitting by separators.

Working:
Typical defaults are around 200 for chunk_size and 0–50 for chunk_overlap. With chunk_size=100 and chunk_overlap=20:
    a. Chunk 1: characters 0–99 (100 chars)
    b. Chunk 2: characters 80–179 (last 20 of chunk 1 + next 80 new chars)
So the last 20 characters of chunk 1 are also the first 20 characters of chunk 2.
More overlap improves context but increases storage and retrieval cost because the same text appears in multiple chunks.

Next up, we'll need to provide an embedding model that we can use to construct our vector store.

In [51]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

Now we can build our in memory QDrant vector store.

In [52]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data",
    embedding=embeddings,
)

We can now add our documents to our vector store.

In [53]:
_ = vector_store.add_documents(documents=split_documents)

Let's define our retriever.

In [54]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

Now we can produce a node for retrieval!

In [55]:
def retrieve(state):
  retrieved_docs = retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

### A - Augmented

Let's create a simple RAG prompt!

In [56]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
You are a helpful assistant who answers questions based on provided context. You must only use the provided context, and cannot use your own knowledge.

### Question
{question}

### Context
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

### G - Generation

We'll also need an LLM to generate responses - we'll use `gpt-4o-nano` to avoid using the same model as our judge model.

In [57]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano")

Then we can create a `generate` node!

In [58]:
def generate(state):
  docs_content = "\n\n".join(doc.page_content for doc in state["context"])
  messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
  response = llm.invoke(messages)
  return {"response" : response.content}

### Building RAG Graph with LangGraph

Let's create some state for our LangGraph RAG graph!

In [59]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class State(TypedDict):
  question: str
  context: List[Document]
  response: str

Now we can build our simple graph!

> NOTE: We're using `add_sequence` since we will always move from retrieval to generation. This is essentially building a chain in LangGraph.

In [60]:
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

Let's do a test to make sure it's doing what we'd expect.

In [61]:
response = graph.invoke({"question" : "What exercises help with lower back pain?"})

In [62]:
response["response"]

'The provided context does not specify particular exercises that help with lower back pain.'

## Task 5: Evaluating our Application with Ragas

Now we can finally do our evaluation!

We'll start by running the queries we generated usign SDG above through our application to get context and responses.

In [63]:
for test_row in dataset:
  response = graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [64]:
dataset.samples[0].eval_sample.response

'To perform a Chest Opener to relieve neck and shoulder tension, clasp your hands behind your back and squeeze.'

Then we can convert that table into a `EvaluationDataset` which will make the process of evaluation smoother.

In [65]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

We'll need to select a judge model - in this case we're using the same model that was used to generate our Synthetic Data.

In [66]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

Next up - we simply evaluate on our desired metrics!

In [67]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
baseline_result

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

{'context_recall': 0.1667, 'faithfulness': 0.4710, 'factual_correctness': 0.5125, 'answer_relevancy': 0.3914, 'context_entity_recall': 0.4147, 'noise_sensitivity_relevant': 0.0684}

## Task 6: Making Adjustments and Re-Evaluating

Now that we've got our baseline - let's make a change and see how the model improves or doesn't improve!




We'll first set our retriever to return more documents, which will allow us to take advantage of the reranking.

In [68]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=30)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data_new_chunks",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data_new_chunks",
    embedding=embeddings,
)

_ = vector_store.add_documents(documents=split_documents)

adjusted_example_retriever = vector_store.as_retriever(search_kwargs={"k": 20})

Reranking, or contextual compression, is a technique that uses a reranker to compress the retrieved documents into a smaller set of documents.

This is essentially a slower, more accurate form of semantic similarity that we use on a smaller subset of our documents.

In [69]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

def retrieve_adjusted(state):
  compressor = CohereRerank(model="rerank-v3.5")
  compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=adjusted_example_retriever, search_kwargs={"k": 5}
  )
  retrieved_docs = compression_retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

We can simply rebuild our graph with the new retriever!

In [70]:
class AdjustedState(TypedDict):
  question: str
  context: List[Document]
  response: str

adjusted_graph_builder = StateGraph(AdjustedState).add_sequence([retrieve_adjusted, generate])
adjusted_graph_builder.add_edge(START, "retrieve_adjusted")
adjusted_graph = adjusted_graph_builder.compile()

In [71]:
response = adjusted_graph.invoke({"question" : "How can I improve my sleep quality?"})
response["response"]

'To improve your sleep quality, consider adopting good sleep hygiene practices such as maintaining a consistent sleep schedule, creating a relaxing bedtime routine (like reading or gentle stretching), and ensuring your bedroom is cool, dark, and quiet. Limit screen exposure 1-2 hours before bed, avoid caffeine after 2 PM, and exercise regularly but not too close to bedtime. Additionally, keep your room temperature between 65-68°F, use blackout curtains or a sleep mask, and ensure your mattress and pillows are comfortable. Incorporating relaxation techniques like meditation, deep breathing, or relaxation exercises can also help. If needed, natural remedies such as herbal teas (chamomile or valerian root) or magnesium supplements may assist, but consult a healthcare provider first.'

In [72]:
import time
import copy

rerank_dataset = copy.deepcopy(dataset)

for test_row in rerank_dataset:
  response = adjusted_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
  time.sleep(10) # To try to avoid rate limiting.

In [73]:
rerank_dataset.samples[0].eval_sample.response

'To perform a Chest Opener and relieve neck and shoulder tension, follow these steps:\n\n1. Clasp your hands behind your back.\n2. Squeeze your shoulder blades together.\n3. Lift your arms slightly upward.\n4. Hold the position for 15 to 30 seconds.\n\nThis exercise helps open up the chest and reduce tension in the neck and shoulders.'

In [74]:
rerank_evaluation_dataset = EvaluationDataset.from_pandas(rerank_dataset.to_pandas())

In [75]:
rerank_result = evaluate(
    dataset=rerank_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
rerank_result

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

{'context_recall': 1.0000, 'faithfulness': 0.7144, 'factual_correctness': 0.7100, 'answer_relevancy': 0.9666, 'context_entity_recall': 0.5440, 'noise_sensitivity_relevant': 0.1301}

### ❓ Question #2:

Which system performed better, on what metrics, and why?

##### Answer:

Rerank wins on 3 metrics (retrieval-focused): context recall, answer relevancy, context entity recall.
Baseline wins on 3 metrics (faithfulness/reliability-focused): faithfulness, factual correctness, noise sensitivity.
Overall, there’s no single winner: they optimize different aspects.

Rerank strengths:
1. Retrieves 20 docs, then reranks to 5 → broader recall, better selection.
2. Higher context_recall and context_entity_recall because more relevant chunks get included.
3. Higher answer_relevancy because retrieved context better matches the query.

Baseline strengths:
1. Simpler retrieval means fewer borderline/off-topic documents.
2. Higher faithfulness and factual_correctness → responses stay closer to the retrieved context.
3. Lower noise_sensitivity → behavior is more stable when extra/irrelevant context is added.

Rerank trade-off:
1. The reranker may surface more diverse chunks; some can introduce conflicting info or distract the LLM.
2. That can slightly hurt faithfulness and factual correctness and increase sensitivity to noise.

### ❓ Question #3:

What are the benefits and limitations of using synthetic data generation for RAG evaluation? Consider both the practical advantages and potential pitfalls.

##### Answer:
Benefits of Synthetic Data for RAG Evaluation
1. Scalability & cost
Can generate many test examples without manual labeling.
Avoids hiring annotators or collecting real user data.
Makes it easier to run frequent evaluations.
2. Ground truth availability
Ground truth answers and reference contexts come from the same source as the questions.
Directly supports reference-based metrics (e.g., context recall, faithfulness).
Fewer labeling inconsistencies than with human-created data.
3. Coverage and diversity
You can target specific document chunks and topics.
Easier to design edge cases, multi-hop reasoning, and rare queries that seldom appear in real logs.
More flexibility to balance difficulty, domain, and query types.
4. Domain adaptation
Useful in domains where real labeled data is scarce or hard to get.
Ragas uses your documents to generate questions tied to your corpus.
5. Reproducibility
Same generator config and documents produce comparable datasets.
Supports before/after comparisons (e.g., baseline vs reranking).
6. Directional insight
As the assignment notes, Ragas is best suited for directional changes.
Absolute scores aren’t always comparable in isolation; relative differences between configurations are more informative.

Limitations & Potential Pitfalls
1. Distribution shift
Synthetic questions may not reflect real user phrasing, typos, ambiguity, or conversational context.
A system that performs well on synthetic data can still fail on production queries.
A system that fails on synthetic data may still be adequate for real use cases.
2. LLM bias in generation
If an LLM generates the test set, it may favor patterns similar to its own output.
Models from the same family may appear “better” on synthetic tests.
Evaluation may not generalize well to other model families or user intents.
3. Circular validation risk
Using the same model for both generation and evaluation can hide certain failure modes.
The generator may miss failure patterns that a different model would trigger.
Using a different model (or model family) for generation can reduce this.
4. Quality variability
Quality depends on the generator prompt and model.
You can get unrealistic or contradictory questions, or questions that don’t match the documents.
Requires inspection and curation of the generated dataset.
5. Misleading absolute scores
A high score on synthetic data does not guarantee good production performance.
Treat synthetic scores as relative signals, not absolute quality guarantees.
Use them to compare configurations, not as stand-alone quality metrics.
6. Missing real-world complexity
Real queries often include follow-ups, implicit context, reformulations, and errors.
Synthetic data tends to be cleaner and may underrepresent these cases.


### ❓ Question #4:

If you were building a production wellness assistant, which Ragas metrics would be most important to optimize for and why? Consider the healthcare/wellness domain specifically.

##### Answer:

Most Important Metrics
1. Faithfulness (highest priority)
    Measures whether all claims in the response can be supported by the retrieved context.
        a. Wellness advice can affect behavior (exercise, diet, sleep, stress).
        b. Hallucinated or unsupported claims (“this herb cures X”) can be harmful.
        c. Strong faithfulness keeps responses tied to your trusted sources and avoids unsupported medical-like claims.
2. Factual Correctness
Checks if the response matches what’s true, not only what’s in the context.
        a. Wellness content often mixes science and pseudoscience.
        b. Wrong or outdated advice can worsen conditions.
        c. Good scores here reduce the chance of confidently stated but incorrect information.
3. Context Recall (LLMContextRecall)
Assesses how much of the needed information is in the retrieved context.
        a. Wellness questions can span multiple topics (diet + exercise + sleep).
        b. Missing relevant chunks can lead to incomplete or misleading advice.
        c. High recall helps ensure key evidence is available for the model.
4. Answer Relevancy
Measures how well the response addresses the user’s question.
        a. Users ask wellness questions in many different ways.
        b. Irrelevant answers waste trust and can misdirect behavior.
        c. Good relevancy keeps responses focused on the user’s actual concern.

Secondary but Still Useful
    1. Context Entity Recall – important when topics, ingredients, conditions, or studies must be represented accurately.
    2. Noise Sensitivity – helps check robustness when retrieval returns some irrelevant context, which could distract the model.

## Activity #1: Implement a Different Reranking Strategy

In this activity, you'll experiment with different reranking parameters or strategies to see how they affect the evaluation metrics.

**Requirements:**
1. Modify the `retrieve_adjusted` function to use different parameters (e.g., change `k` values, try different top_n for reranking)
2. Or implement a different retrieval enhancement strategy (e.g., hybrid search, query expansion)
3. Run the evaluation and compare results with the baseline and reranking results above
4. Document your findings in the markdown cell below

In [78]:
### YOUR CODE HERE ###

# Implement your custom retrieval strategy here
# Example: modify retrieve_adjusted with different parameters

def retrieve_custom(state):
    """Custom retrieval: more candidates (k=30) before reranking, top_n=8 after reranking."""
    compressor = CohereRerank(model="rerank-v3.5", top_n=8)
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=adjusted_example_retriever,
        search_kwargs={"k": 30},
    )
    retrieved_docs = compression_retriever.invoke(state["question"])
    return {"context": retrieved_docs}


custom_graph_builder = StateGraph(AdjustedState).add_sequence([retrieve_custom, generate])
custom_graph_builder.add_edge(START, "retrieve_custom")
custom_graph = custom_graph_builder.compile()

response = custom_graph.invoke({"question" : "How can I improve my sleep quality?"})
response["response"]

"To improve your sleep quality, you can adopt several sleep hygiene practices and create an optimal sleep environment. Here are some recommendations based on the provided information:\n\n1. Maintain a consistent sleep schedule by going to bed and waking up at the same times every day, even on weekends.\n2. Create a relaxing bedtime routine, such as reading a paper book, gentle stretching, or taking a warm bath, to signal to your body that it's time to sleep.\n3. Keep your bedroom cool, dark, and quiet. Use blackout curtains or a sleep mask to block out light, and consider white noise machines or earplugs to reduce noise.\n4. Limit screen exposure 1-2 hours before bed by putting away electronic devices, as screens can interfere with sleep.\n5. Avoid caffeine after 2 PM and heavy meals or alcohol before bed, as these can disrupt sleep.\n6. Exercise regularly, but not too close to bedtime, to promote better sleep.\n7. Ensure your mattress and pillows are comfortable to create a cozy sleep

In [79]:
# Run custom strategy over full dataset (may take a while due to rate limiting)
custom_dataset = copy.deepcopy(dataset)

for test_row in custom_dataset:
    response = custom_graph.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"]
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
    time.sleep(10)  # To try to avoid rate limiting

In [80]:
custom_evaluation_dataset = EvaluationDataset.from_pandas(custom_dataset.to_pandas())

custom_result = evaluate(
    dataset=custom_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config,
)
custom_result

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

{'context_recall': 1.0000, 'faithfulness': 0.7116, 'factual_correctness': 0.7783, 'answer_relevancy': 0.8940, 'context_entity_recall': 0.4954, 'noise_sensitivity_relevant': 0.0653}

In [81]:
# Quick comparison: baseline vs reranking vs custom
print("Baseline:", baseline_result)
print("\nReranking (k=5, top_n default):", rerank_result)
print("\nCustom (k=30, top_n=8):", custom_result)

Baseline: {'context_recall': 0.1667, 'faithfulness': 0.4710, 'factual_correctness': 0.5125, 'answer_relevancy': 0.3914, 'context_entity_recall': 0.4147, 'noise_sensitivity_relevant': 0.0684}

Reranking (k=5, top_n default): {'context_recall': 1.0000, 'faithfulness': 0.7144, 'factual_correctness': 0.7100, 'answer_relevancy': 0.9666, 'context_entity_recall': 0.5440, 'noise_sensitivity_relevant': 0.1301}

Custom (k=30, top_n=8): {'context_recall': 1.0000, 'faithfulness': 0.7116, 'factual_correctness': 0.7783, 'answer_relevancy': 0.8940, 'context_entity_recall': 0.4954, 'noise_sensitivity_relevant': 0.0653}


### Activity #1 Findings:

*Document your findings here: What strategy did you try? How did it compare to the baseline and reranking results?*

Context_recal remained same, faithfullness was slightly lower.
factual_correctness was higher, other metrcis were lower.

Factual correctness improved: Custom sends more context, so more factual details are available. The model can better support its claims with evidence.

Noise sensitivity (0.13 → 0.07)
Noise sensitivity measures how robust the system is when some irrelevant context is present.
    1. Lower score suggests the system is more affected by noisy/irrelevant context.
    2. 8 chunks create more opportunities for weak or off-topic content.
    3. That extra context makes the model more sensitive to noise, so the score goes down.

Answer relevancy (0.97 → 0.89)
    1. The model sees more, sometimes less relevant, material.
    2. It can mix in tangential content from ranks 4–8.
    3. Answers become less tightly aligned with the question.